In [65]:
import customfunctions as ctf
import pandas as pd
import numpy as np
import scipy.signal as signal
import heartpy as hp
import neurokit2 as nk
import prepro as prep  # Custom functions for preprocessing
import matplotlib.pyplot as plt
import os
import glob
from pathlib import Path
from scipy.ndimage import uniform_filter1d

# Parameters

### Preprocessing parameters

In [66]:
# BVP preprocessing
bvp_prepro_winsz = 10
bvp_prepro_ovlap = 0.02
bvp_fs = 64

# EDA preprocessing
eda_fs = 4
iter_num = 7

### Processing parameters

In [67]:
basalscene = 0
targetscene = 4
# Note, subject 10 does not have scene 2 nor 3 data
subjects = [1, 2, 3, 4, 6, 8, 9, 10]
winsz = 60
ovlap = 0.5
base_dir = Path().resolve()

# Data import function

In [ ]:
folderpath = os.path.join(base_dir, "..", "RawData")
folderpath = os.path.abspath(folderpath)


def dataimport(subjects, folderpath, scene):
    bvp_rawdata_list = []
    eda_rawdata_list = []

    for subject in subjects:
        subject_str = f"S{subject}" if subject == 10 else f"S{subject:02d}"
        pattern = os.path.join(
            folderpath,
            f"S{subject}",
            "Empatica",
            f"P300_{subject_str}R0{scene}*",
            "Raw",
        )
        matched_folders = glob.glob(pattern)
        raw_folder = matched_folders[0]
        bvp_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileBVP.csv")).drop(
            "Datetime", axis=1
        )
        eda_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileEDA.csv")).drop(
            "Datetime", axis=1
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_rawdata_list.append(bvp_dataholder)
        eda_rawdata_list.append(eda_dataholder)

    bvp_rawdata_df = pd.concat(bvp_rawdata_list, ignore_index=True)
    eda_rawdata_df = pd.concat(eda_rawdata_list, ignore_index=True)

    return bvp_rawdata_df, eda_rawdata_df


basal_bvp_rawdata, basal_eda_rawdata = dataimport(
    subjects, folderpath, scene=basalscene
)
tscene_bvp_rawdata, tscene_eda_rawdata = dataimport(
    subjects, folderpath, scene=targetscene
)

1 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S1\\Empatica\\P300_S01R00_29042024_1237\\Raw']
2 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S2\\Empatica\\P300_S02R00_29042024_1342\\Raw']
3 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S3\\Empatica\\P300_S03R00_29042024_1628\\Raw']
4 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S4\\Empatica\\P300_S04R00_30042024_0945\\Raw']
6 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S6\\Empatica\\P300_S06R00_30042024_1632\\Raw']
8 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S8\\Empatica\\P300_S08R00_02052024_1206\\Raw']
9 ['C:\\Users\\josee\\OneDrive\\Documentos\\GitHub\\neurohumanities-lab\\OfflineProcessing\\RawData\\S9\\Empatic

# Preprocessing function

In [69]:
def preprocess(subjects, bvp_df, eda_df, bvp_fs, eda_fs, winsz, ovlap, iter_num):
    bvp_prepdata_list = []
    eda_prepdata_list = []
    for subject in subjects:
        bvp_dataholder = pd.DataFrame(
            prep.preprocess_bvp(
                sig=bvp_df[bvp_df["Subject"] == subject]["valueBVP"],
                fs=bvp_fs,
                winsz=winsz,
                ovlap=ovlap,
            ),
            columns=["valueBVP"],
        )
        eda_dataholder = pd.DataFrame(
            prep.preprocess_eda(
                sig=eda_df[eda_df["Subject"] == subject]["valueEDA"],
                fs=eda_fs,
                iter_num=iter_num,
                verbose=False,
            )[0],
            columns=["valueEDA"],
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_prepdata_list.append(bvp_dataholder)
        eda_prepdata_list.append(eda_dataholder)

    bvp_prepdata_df = pd.concat(bvp_prepdata_list, ignore_index=True)
    eda_prepdata_df = pd.concat(eda_prepdata_list, ignore_index=True)

    return bvp_prepdata_df, eda_prepdata_df


basal_bvp_prepdata, basal_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=basal_bvp_rawdata,
    eda_df=basal_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)

tscene_bvp_prepdata, tscene_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=tscene_bvp_rawdata,
    eda_df=tscene_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)


# Processing function

In [70]:
import warnings
def get_bvp_feats(window, smooth_window, fs):
    wd = {}
    wd = hp.peakdetection.detect_peaks(
        window, smooth_window, ma_perc=20, sample_rate=fs
    )
    wd = hp.analysis.calc_rr(wd["peaklist"], sample_rate=fs, working_data=wd)
    wd = hp.peakdetection.check_peaks(
        wd["RR_list"], wd["peaklist"], window[wd["peaklist"]], working_data=wd
    )
    wd = hp.analysis.clean_rr_intervals(working_data=wd, method="quotient-filter")
    rr_list = wd["RR_list_cor"]
    rr_diff = np.diff(rr_list)
    rr_sqdiff = np.power(rr_diff, 2)
    wd, msrs = hp.analysis.calc_ts_measures(
        rr_list, rr_diff, rr_sqdiff, working_data=wd
    )
    wd, msrs = hp.analysis.calc_fd_measures(measures=msrs, working_data=wd)
    bvp_feats = pd.DataFrame([msrs])[
        [
            "bpm",
            "sdnn",
            'rmssd',
            "pnn50",
            "hr_mad",
            "lf",
            "hf",
            "lf/hf",
            "p_total",
            "lf_nu",
            "hf_nu",
        ]
    ]

    return bvp_feats


def get_eda_feats(window, fs):
    signals, info = nk.eda_process(window, fs)
    mean_eda = np.nanmean(window)
    mean_tonic = np.nanmean(signals["EDA_Tonic"])
    scr_peak_count = np.nansum(signals["SCR_Peaks"])
    scr_sum_amp = np.nansum(info["SCR_Amplitude"])
    scr_mean_amp = np.nanmean(info["SCR_Amplitude"])
    scr_mean_risetime = np.nanmean(info["SCR_RiseTime"])
    scr_mean_recoverytime = np.nanmean(info["SCR_RecoveryTime"])

    eda_feats = pd.DataFrame(
        [
            {
                "mean_eda": mean_eda,
                "mean_tonic": mean_tonic,
                "scr_peak_count": scr_peak_count,
                "scr_sum_amp": scr_sum_amp,
                "scr_mean_amp": scr_mean_amp,
                "scr_mean_risetime": scr_mean_risetime,
                "scr_mean_recoverytime": scr_mean_recoverytime,
            }
        ]
    )

    return eda_feats


def process(subjects, bvp_prepdata_df, eda_prepdata_df, bvp_fs, eda_fs, ovlap, winsz):
    feat_mat_list = []
    for subject in subjects:
        # BVP
        bvp_probe = np.array(
            bvp_prepdata_df[bvp_prepdata_df["Subject"] == subject]["valueBVP"]
        )
        bvp_smooth = uniform_filter1d(
            bvp_probe, size=int(0.75 * bvp_fs), mode="nearest"
        )
        bvp_windowed = ctf.timewindowpadded(
            data=bvp_probe, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        smooth_windowed = ctf.timewindowpadded(
            data=bvp_smooth, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        bvp_holder_list = []
        for windnum in range(0,bvp_windowed.shape[0]):
            bvp_feat_row = get_bvp_feats(
                window=bvp_windowed[windnum],
                smooth_window=smooth_windowed[windnum],
                fs=bvp_fs,
            )
            bvp_holder_list.append(bvp_feat_row)
        bvp_holder_df = pd.concat(bvp_holder_list, ignore_index=True)

        # EDA
        eda_probe = np.array(
            eda_prepdata_df[eda_prepdata_df["Subject"] == subject]["valueEDA"]
        )
        eda_windowed = ctf.timewindowpadded(
            data=eda_probe, fs=eda_fs, ovlap=ovlap, winsz=winsz
        )
        eda_holder_list = []
        for windnum in range(0,eda_windowed.shape[0]):
            eda_feat_row = get_eda_feats(window=eda_windowed[windnum], fs=eda_fs)
            eda_holder_list.append(eda_feat_row)
        eda_holder_df = pd.concat(eda_holder_list, ignore_index=True)

        valid_len = min(len(bvp_holder_df), len(eda_holder_df))
        joined_df = pd.concat([bvp_holder_df.iloc[:valid_len], eda_holder_df.iloc[:valid_len]], axis=1)
        joined_df.insert(loc=0, column="Subject", value=[subject] * len(joined_df))
        feat_mat_list.append(joined_df)
        
    feat_mat_df = pd.concat(feat_mat_list, ignore_index=True)
    return feat_mat_df


basal_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=basal_bvp_prepdata,
    eda_prepdata_df=basal_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

tscene_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=tscene_bvp_prepdata,
    eda_prepdata_df=tscene_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

warnings.filterwarnings('ignore')

In [71]:
basal_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,60.826446,53.793024,63.233543,0.155556,31.2500,924.116936,1170.473541,0.789524,2.094590e+03,44.119218,55.880782,-0.008248,-0.008088,10,0.091262,0.009126,0.425000,0.678571
1,1,63.605067,74.064890,75.324817,0.160000,46.8750,1409.887040,930.381341,1.515386,2.340268e+03,60.244673,39.755327,-0.001568,-0.001620,13,0.078395,0.006030,0.423077,0.642857
2,1,63.691194,56.350347,58.694936,0.148148,31.2500,686.325269,826.141762,0.830760,1.762398e+03,45.377866,54.622134,-0.000471,-0.000526,11,0.095344,0.008668,0.545455,0.750000
3,1,67.467057,122.776171,115.788412,0.127660,62.5000,1975.783853,1136.872088,1.737912,3.112656e+03,63.475819,36.524181,-0.000456,-0.000566,12,0.080819,0.007347,1.250000,1.375000
4,1,71.831645,106.115797,116.904579,0.127660,85.9375,1485.146287,984.318328,1.508807,2.469465e+03,60.140416,39.859584,0.003984,0.003746,22,0.079369,0.003779,0.916667,0.750000
5,1,66.804951,62.836413,55.995817,0.156863,46.8750,579.078261,1834.289281,0.315696,2.413368e+03,23.994615,76.005385,0.014380,0.014300,12,0.111295,0.009275,0.770833,1.111111
6,2,68.534343,52.880012,45.780491,0.092308,31.2500,292.261292,412.743283,0.708095,8.482538e+02,41.455233,58.544767,-0.036505,-0.036688,56,0.330408,0.006007,0.422727,0.500000
7,2,73.079585,90.994130,62.379692,0.123077,46.8750,1996.809797,1346.703264,1.482739,4.498593e+03,59.721908,40.278092,-0.006954,-0.007078,57,0.326441,0.005829,0.370536,0.527778
8,2,74.517890,82.447269,58.595817,0.079365,46.8750,385.864356,757.807670,0.509185,3.164651e+03,33.739074,66.260926,0.005557,0.005445,51,0.293340,0.005752,0.450980,0.583333
9,2,76.072351,74.892770,43.580620,0.029412,62.5000,820.606860,348.138072,2.357130,1.608311e+03,70.212656,29.787344,0.003289,0.003127,51,0.330304,0.006477,0.450980,0.576923


In [72]:
tscene_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,80.457143,163.006056,191.153639,0.428571,140.6250,3.172751e+03,1.281359e+04,0.247608,1.598634e+04,19.846642,80.153358,-0.064255,-0.066151,6,0.651519,0.130304,0.450000,0.375000
1,1,82.997118,137.342081,210.088403,0.428571,109.3750,0.000000e+00,6.245153e+03,0.000000,6.245153e+03,0.000000,100.000000,-0.000356,-0.000200,5,1.180030,0.236006,0.450000,NaN
2,1,91.428571,193.228465,311.633844,0.285714,171.8750,0.000000e+00,1.219651e+04,0.000000,1.219651e+04,0.000000,100.000000,-0.006593,-0.005168,6,0.711107,0.118518,0.875000,0.750000
3,1,90.423862,145.557775,216.948899,0.285714,125.0000,0.000000e+00,3.645333e+03,0.000000,3.645333e+03,0.000000,100.000000,-0.000781,-0.000967,9,0.135318,0.015035,1.166667,0.916667
4,1,100.800000,155.944997,208.463501,0.350000,93.7500,0.000000e+00,8.326933e+03,0.000000,8.326933e+03,0.000000,100.000000,0.001036,0.001210,11,0.167177,0.015198,0.863636,1.468750
5,1,81.355932,141.421356,182.524033,0.375000,140.6250,1.728348e+03,3.353294e+03,0.515418,5.081642e+03,34.011600,65.988400,-0.011342,-0.010631,11,0.303391,0.027581,0.772727,1.062500
6,2,78.837650,61.514043,66.892749,0.140625,31.2500,8.294160e+02,8.914498e+02,0.930412,1.720866e+03,48.197599,51.802401,0.013959,0.014282,36,0.288217,0.008235,0.521429,0.634615
7,2,77.796771,80.830459,80.108743,0.222222,62.5000,1.124317e+03,1.324564e+03,0.848820,2.448880e+03,45.911459,54.088541,0.020214,0.020313,3,1.203188,0.401063,0.333333,1.250000
8,2,75.730858,81.633482,69.169556,0.164179,54.6875,1.908573e+03,8.530462e+02,2.237362,4.208119e+03,69.110652,30.889348,0.036361,0.036194,5,1.779073,0.355815,0.450000,1.250000
9,2,74.366197,56.509579,51.018768,0.138462,39.0625,1.237454e+03,3.443531e+02,3.593561,1.861208e+03,78.230396,21.769604,0.007653,0.007455,5,0.630056,0.126011,0.450000,6.250000


In [73]:
def norm_with_ref(subjects, feat_mat, ref_mat):
        # This line is redundant, eliminate and substitute application on this cell based on preference.
    channel_cols = [
        col for col in ref_mat.columns if col not in ["Subject"]
    ]

    normdata_list = []

    # Data is normalized per subject
    for subject in subjects:
        data = feat_mat[feat_mat["Subject"] == subject].copy()
        # Baseline normalization
        ref_data = ref_mat[ref_mat["Subject"] == subject].drop(
            "Subject", axis=1
        )
        ref_mean = np.nanmean(ref_data)
        ref_std = np.nanstd(ref_data)
        data[channel_cols] = (data[channel_cols] - ref_mean) / ref_std

        normdata_list.append(data)
    normdata_df = pd.concat(normdata_list, ignore_index=True).fillna(0)

    return normdata_df

norm = norm_with_ref(subjects=subjects, feat_mat=tscene_feat_mat, ref_mat=basal_feat_mat)
norm

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,-0.314758,-0.186063,-0.142181,-0.439523,-0.220956,4.506155e+00,1.953630e+01,-0.439805,2.448265e+01,-0.409250,-0.315232,-0.440291,-0.440294,-0.430837,-0.439176,-0.439988,-0.439490,-0.439607
1,1,-0.310798,-0.226074,-0.112662,-0.439523,-0.269675,-4.401913e-01,9.296058e+00,-0.440191,9.296058e+00,-0.440191,-0.284290,-0.440192,-0.440192,-0.432396,-0.438352,-0.439823,-0.439490,0.000000
2,1,-0.297653,-0.138946,0.045649,-0.439746,-0.172237,-4.401913e-01,1.857427e+01,-0.440191,1.857427e+01,-0.440191,-0.284290,-0.440202,-0.440199,-0.430837,-0.439083,-0.440007,-0.438827,-0.439022
3,1,-0.299220,-0.213265,-0.101966,-0.439746,-0.245315,-4.401913e-01,5.242914e+00,-0.440191,5.242914e+00,-0.440191,-0.284290,-0.440193,-0.440193,-0.426160,-0.439980,-0.440168,-0.438372,-0.438762
4,1,-0.283043,-0.197072,-0.115195,-0.439646,-0.294034,-4.401913e-01,1.254157e+01,-0.440191,1.254157e+01,-0.440191,-0.284290,-0.440190,-0.440189,-0.423042,-0.439931,-0.440168,-0.438845,-0.437902
5,1,-0.313357,-0.219714,-0.155635,-0.439607,-0.220956,2.254318e+00,4.787624e+00,-0.439388,7.482133e+00,-0.387167,-0.337315,-0.440209,-0.440208,-0.423042,-0.439718,-0.440148,-0.438987,-0.438535
6,2,-0.256594,-0.278118,-0.271435,-0.354372,-0.315720,6.759612e-01,7.530351e-01,-0.353390,1.783542e+00,-0.294663,-0.290184,-0.354529,-0.354528,-0.309818,-0.354188,-0.354536,-0.353898,-0.353758
7,2,-0.257888,-0.254118,-0.255015,-0.354270,-0.276893,1.042360e+00,1.291157e+00,-0.353492,2.688064e+00,-0.297504,-0.287344,-0.354521,-0.354521,-0.350819,-0.353051,-0.354048,-0.354132,-0.352993
8,2,-0.260454,-0.253121,-0.268607,-0.354342,-0.286600,2.016759e+00,7.053205e-01,-0.351766,4.873828e+00,-0.268680,-0.316168,-0.354501,-0.354501,-0.348334,-0.352336,-0.354104,-0.353987,-0.352993
9,2,-0.262150,-0.284336,-0.291158,-0.354374,-0.306013,1.182928e+00,7.329508e-02,-0.350081,1.957910e+00,-0.257349,-0.327499,-0.354537,-0.354537,-0.348334,-0.353763,-0.354390,-0.353987,-0.346781


In [74]:
outpath = os.path.join(base_dir, 'Outputs')

norm.to_csv(os.path.join(outpath, rf'biometric_feat_mat_scene_{targetscene}.csv'), index=False)